In [1]:
import pandas as pd
from embeders import SentenceTransformerEmbeddingFunction

e:\repos\pessoal\redem-index\studies\fake_news\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_pubs = pd.read_excel("./2023_Completo_redem_0304.xlsx")

In [3]:
df = pd.read_csv("df_res.csv")
print(f"Total de mensagens: {len(df)}")
# df[df["min_distance"] > 0].sort_values("min_distance").head(10).reset_index().merge(
#     df_pubs[["Message-ID", "Profile", "partido", "bloco"]],
#     left_on="pub_id",
#     right_on="Message-ID",
#     how="left",
# )

Total de mensagens: 91648


In [4]:
# idx = 37017
# # Acertos: 10185, 9199, 9857, 8460, 9272,  2513,   9563,9438, 46118
# # tau = 0.307673
# # Falsos positivos: 2513, 67018, 9863
# # Talvez: 37017, 21542, 27389, 61591

# print("POSTAGEM:")
# print(df.iloc[idx]["message"])
# print(
#     f"Autor: {df_pubs[df_pubs['Message-ID'] == df.iloc[idx]['pub_id']]['Profile'].values[0]}"
# )
# print("=" * 100)
# print("DOCUMENTO MAIS SIMILAR:")
# print(df.iloc[idx]["min_dist_document"])

In [5]:
# pd.set_option("display.max_rows", None)

# df_rank = (
#     df.merge(df_pubs, left_on="pub_id", right_on="Message-ID", how="left")
#     .groupby("Profile")
#     .agg({"partido": "first", "pub_id": "count", "min_distance": "min"})
#     .sort_values("min_distance")
# )
# df_rank.head(10)

# Com reescrita

In [6]:
import chromadb
import pandas as pd
from embeders import SentenceTransformerEmbeddingFunction

In [7]:
client = chromadb.PersistentClient(path=r".chroma_db/")

# Use Ollama for both add and query embeddings
# ef = OllamaEmbeddingFunction(model="mxbai-embed-large", host="http://127.0.0.1:11434")
ef = SentenceTransformerEmbeddingFunction('sentence-transformers/all-MiniLM-L6-v2')

collection = client.get_or_create_collection(
    name="checks_v2_rewrite_cosine_sentence_transformer",
    embedding_function=ef,
    metadata={"hnsw:space": "cosine"},
)


In [8]:
df_rewrites = pd.read_excel("./Analise manual Aos fatos_v2.xlsx")
df_rewrites = df_rewrites.dropna(subset=["resumo_2"])

In [9]:
collection.upsert(
    ids=[str(i) for i in df_rewrites.index.tolist()],
    documents=df_rewrites["resumo_2"].to_list(),
    metadatas=df_rewrites[["Ano", "Título", "tags"]].to_dict(orient="records")
)


## Compare

In [10]:
from tqdm import tqdm

In [15]:
df = df.sort_values("min_distance")

batch_size = 32  # You can change this to the batch size you want
messages = df["message"].tolist()
num_batches = (len(messages) + batch_size - 1) // batch_size

for batch_idx in tqdm(range(num_batches)):
    start = batch_idx * batch_size
    end = min((batch_idx + 1) * batch_size, len(messages))
    batch_indices = df.index[start:end]
    batch_messages = messages[start:end]

    res = collection.query(
        query_texts=batch_messages,
        n_results=3
    )

    for i, idx in enumerate(batch_indices):
        df.at[idx, "resumo_2"] = res["documents"][i][0] if res["documents"][i] else None
        df.at[idx, "min_distance_2"] = res["distances"][i][0] if res["distances"][i] else None
        df.at[idx, "all_docs_2"] = str(res["documents"][i]) if res["documents"][i] else None
        df.at[idx, "all_distances_2"] = str(res["distances"][i]) if res["distances"][i] else None

100%|██████████| 2864/2864 [1:06:10<00:00,  1.39s/it]


In [17]:
df.to_excel("df_res_2.xlsx")

In [27]:
for i, df_row in enumerate(df.sort_values("all_distances_2").iterrows()):
    idx, row = df_row
    print(row["message"])
    print("-" * 100)
    print(f"v1: {row['min_distance']} - {row['min_dist_document']}")
    print("-" * 100)
    print(f"v2: {row['min_distance_2']} - {row['resumo_2']}")
    print(f"v2: {row['all_distances_2']} - {row['all_docs_2']}")
    print("=" * 100)
    if row['min_distance_2'] > .4:
        break


Mauro Cid falsificou certificados de vacina contra a Covid-19 a pedido de Bolsonaro, segundo delação feita à Polícia Federal! A versão contraria o depoimento do ex-presidente que disse desconhecer a fraude.

Como se adulterar documentos do Ministério da Saúde já não fosse grave, a ordem de Bolsonaro, em meio a uma pandemia global, é criminosa! Que todos os envolvidos sejam devidamente responsabilizados.
----------------------------------------------------------------------------------------------------
v1: 0.3310762941837311 - O deputado André Janones (Avante-MG) afirmou nas redes sociais que Jair Bolsonaro foi condenado nos EUA e investigado pelo FBI por fraudar o cartão de vacinação contra Covid-19, mas não há provas dessa alegação. Bolsonaro é investigado no Brasil pela PF, com autorização do ministro Alexandre de Moraes, por suposta falsificação do documento para facilitar sua viagem aos EUA. A operação também atingiu aliados, como Mauro Cid, preso na ocasião. Não há indícios de pa

# Rerank

In [28]:
from sentence_transformers  import CrossEncoder

# model = CrossEncoder("zeroentropy/zerank-1", trust_remote_code=True)
model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')


In [29]:
for i, df_row in enumerate(df.iterrows()):
    idx, row = df_row
    print(row["message"])
    print("-" * 100)
    print(f"v1: {row['min_distance']} - {row['min_dist_document']}")
    print("-" * 100)
    print(f"v2: {row['min_distance_2']} - {row['resumo_2']}")
    print(f"v2: {row['all_distances_2']} - {row['all_docs_2']}")
    print(f"-" * 100)

    list_docs_v1 = row['all_documents'].replace("[", "").replace("]", "").split("', '")
    query_docs_v1 = [(row["message"], d) for d in list_docs_v1]

    list_docs_v2 = row['all_docs_2'].replace("[", "").replace("]", "").split("', '")
    query_docs_v2 = [(row["message"], d) for d in list_docs_v2]

    scores_v1 = model.predict(query_docs_v1)
    scores_v2 = model.predict(query_docs_v2)

    df.at[idx, "scores_v1"] = str(scores_v1)
    df.at[idx, "scores_v2"] = str(scores_v2)

    print("RESULTADOS DO RERANK V1:")
    for i, _ in enumerate(query_docs_v1):
        print(f"{scores_v1[i]} --> {list_docs_v1[i]}")

    print("RESULTADOS DO RERANK V2:")
    for i, _ in enumerate(query_docs_v2):
        print(f"{scores_v2[i]} --> {list_docs_v2[i]}")
    print("=" * 100)
    if i == 20:
        break

A que ponto chegamos quando é necessário destacar na Lei de Diretrizes Orçamentárias que a União não pode realizar despesas com ações que incentivem a invasão de propriedade privada, promovam opções sexuais diferentes do sexo biológico para crianças e adolescentes, ou que diminuam ou desconstruam o conceito de família tradicional. Sem contar o absurdo da cirurgia de mudança de sexo em crianças e adolescentes. Vamos continuar lutando sempre pelo que é certo.

#plnacional22 #riograndedosul #patriotas #pl22rs #camaradosdeputados  #direitapatriota
----------------------------------------------------------------------------------------------------
v1: 0.2486626505851745 - Três parlamentares — Gayer, Valadares e Márcio Marinho (Republicanos-BA) — também publicaram no Instagram votos pela proibição da “mudança de sexo em crianças com verba da União”. A inclusão dessa vedação na Lei de Diretrizes Orçamentárias, no entanto, é baseada em desinformação, uma vez que crianças e adolescentes não pod

KeyboardInterrupt: 

In [32]:
row

message              Enquanto a cidade do Rio de Janeiro e diversas...
min_distance                                                  0.326392
min_dist_document    Depois da eleição de Lula, Gayer passou a defe...
doc_id                                                               9
pub_id                                               18393087904062480
all_distances        [0.32639193534851074, 0.4065541923046112, 0.43...
all_documents        ['Depois da eleição de Lula, Gayer passou a de...
all_ids                                               ['9', '0', '10']
resumo_2                                                           NaN
min_distance_2                                                     NaN
all_docs_2                                                         NaN
all_distances_2                                                    NaN
scores_v1                                                          NaN
scores_v2                                                          NaN
Name: 

In [15]:
query_docs_v1

[('A que ponto chegamos quando é necessário destacar na Lei de Diretrizes Orçamentárias que a União não pode realizar despesas com ações que incentivem a invasão de propriedade privada, promovam opções sexuais diferentes do sexo biológico para crianças e adolescentes, ou que diminuam ou desconstruam o conceito de família tradicional. Sem contar o absurdo da cirurgia de mudança de sexo em crianças e adolescentes. Vamos continuar lutando sempre pelo que é certo.\n\n#plnacional22 #riograndedosul #patriotas #pl22rs #camaradosdeputados  #direitapatriota',
  '['),
 ('A que ponto chegamos quando é necessário destacar na Lei de Diretrizes Orçamentárias que a União não pode realizar despesas com ações que incentivem a invasão de propriedade privada, promovam opções sexuais diferentes do sexo biológico para crianças e adolescentes, ou que diminuam ou desconstruam o conceito de família tradicional. Sem contar o absurdo da cirurgia de mudança de sexo em crianças e adolescentes. Vamos continuar lut